In [ ]:
from pathlib import Path
import os
import sys
import warnings
import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scdesigner" / "src").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "scdesigner" / "src"))

from scdesigner.simulators import NegBinCopula

N_CELLS = 1000
N_GENES = 200
SEED = 20260506
DATA_PATH = REPO_ROOT / "data" / "HVG_embryoatlas.h5ad"

rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)

adata = ad.read_h5ad(DATA_PATH)
cell_idx = rng.choice(adata.n_obs, size=min(N_CELLS, adata.n_obs), replace=False)
adata = adata[cell_idx, :].copy()

X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
expressed_genes = np.flatnonzero(np.asarray(X.mean(axis=0)).ravel() > 0)
gene_idx = rng.choice(expressed_genes, size=min(N_GENES, len(expressed_genes)), replace=False)
adata = adata[:, np.sort(gene_idx)].copy()

sim = NegBinCopula(mean_formula = "~ celltype + stage",
                   dispersion_formula = "~ 1")
sim.fit(adata)

In [ ]:
import matplotlib.pyplot as plt

history = sim.marginal.fit_history_df
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

In [ ]:
from scdesigner.simulators import ZeroInflatedNegBinCopula

sim = ZeroInflatedNegBinCopula(mean_formula = "~ celltype + stage",
                               dispersion_formula = "~ 1",
                                 zero_inflation_formula = "~ 1")
sim.fit(adata)

history = sim.marginal.fit_history_df
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

In [ ]:
from scdesigner.datasets import pancreas

example_sce = pancreas()

sim = NegBinCopula(mean_formula="~ pseudotime",
                   dispersion_formula="~ pseudotime",
                   copula_formula="~ -1 + cell_type")
# Train with larger max_epochs, lr and lower loss_tol for complicated mdoels
sim.fit(example_sce, max_epochs=800, lr = 0.02, loss_tol = 1e-5)
history = sim.marginal.fit_history_df
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")

In [ ]:
sim = ZeroInflatedNegBinCopula(mean_formula="~ pseudotime",
                   dispersion_formula="~ pseudotime",
                    zero_inflation_formula = "~ 1",
                   copula_formula="~ -1 + cell_type")
# Train with larger max_epochs, lr and lower loss_tol for complicated mdoels
sim.fit(example_sce, max_epochs=800, lr = 0.02, loss_tol = 1e-5)
history = sim.marginal.fit_history_df
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("Epoch")
plt.ylabel("Loss")